# Agent Foundations

### Reasoning Loops, Tool Calling & Raw Python Agents

Raw Python agent built directly on an LLM API via **Groq** , no LangChain, no LangGraph.

Groq provides an OpenAI-compatible API, so we use the `openai` Python SDK with Groq's API endpoint. The agent manually implements the tool-calling loop: the model decides whether a tool is needed, Python executes the selected tool, the result is returned to the model, and the process repeats until a final answer is produced.

**Setup required before running:**

```bash
pip install openai python-dotenv
```

Create a `.env` file in the same folder as this notebook containing:

```text
GROQ_API_KEY=your-key-here
```

**Model:**

```python
MODEL = "openai/gpt-oss-20b"
```

The notebook uses Groq's OpenAI-compatible endpoint:

```python
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=api_key,
)
```

No agent framework is used. The ReAct-style loop, tool dispatching, error handling, working memory, and iteration safeguard are implemented manually in Python.


In [17]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# api_key = os.getenv("OPENROUTER_API_KEY")
api_key=os.getenv("GROQ_API_KEY")

print("API key loaded:", api_key is not None)

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",

    api_key=api_key,
)

MODEL = "openai/gpt-oss-20b"


API key loaded: True


## Task 1: Agent Concepts & Mental Model

**Chatbot vs Workflow vs Agent**

- **Chatbot**: single request → single response. No memory of intent beyond the conversation
  turn, no tool use, no planning. It answers; it does not act.
- **Workflow** (a.k.a. "pipeline" or "script"): a FIXED sequence of steps decided by the
  developer ahead of time, e.g. *"extract text → summarize → send email."* The steps and their
  order never change at runtime, regardless of what the data looks like. Reliable and
  predictable, but brittle if reality doesn't match the assumptions baked into the sequence.
- **Agent**: the MODEL decides, at runtime, which steps to take, which tools to call, in what
  order, and when it's done — based on what it observes after each action. The control flow
  lives inside the loop, not in the developer's code.

**What makes something "agentic"?**

1. **Autonomy** — the model chooses actions, not just answers a prompt.
2. **Tool use** — it can affect / query the outside world (APIs, files, code).
3. **Multi-step planning** — it can break a goal into a sequence of sub-actions.
4. **Self-correction** — it can notice a bad result (tool error, wrong output) and change its
   next action instead of blindly continuing.

If a system has *only* tool use but no autonomy over sequencing (e.g. a single tool call then
stop), it's closer to a "tool-augmented chatbot" than an agent. The defining trait is the
**loop**: the model's own output decides what happens next.

**ReAct pattern — Reason → Act → Observe → repeat**

```
messages = [user_task]
while not done:
    response = model(messages)                    # REASON: model thinks, decides next move
    if response.wants_tool:
        result = execute_tool(response.tool_call)  # ACT
        messages.append(tool_call)
        messages.append(result)                    # OBSERVE
    else:
        done = True                                # model produced final answer
return response.final_answer
```

```
┌─────────┐     ┌────────┐     ┌─────────┐
│ REASON  │ --> │  ACT   │ --> │ OBSERVE │ --┐
│ (LLM    │     │ (call  │     │ (tool   │   │
│ decides)│     │ tool)  │     │ result) │   │
└─────────┘     └────────┘     └─────────┘   │
     ^                                       │
     └───────────────────────────────────────┘
          repeat until model has enough info
               to give a final answer
```

**When is an agent overkill?**

An agent is overkill when the task has a known, fixed sequence of steps that never needs to
branch based on intermediate results — a plain script or a single well-crafted prompt does the
job faster, cheaper, and more predictably. Reach for an agent only when the *number or order* of
steps genuinely depends on what happens along the way, not just because tool use is involved.


## Task 2: Tool Calling Fundamentals

Groq provides an **OpenAI-compatible tool-calling interface**, so the tools use the OpenAI-style function schema:

```json
{
  "type": "function",
  "function": {
    "name": "...",
    "description": "...",
    "parameters": { ... }
  }
}
```

The `parameters` field contains a standard **JSON Schema** describing the arguments that the model may provide to the tool. Clear tool names, descriptions, and parameter schemas help the model select the correct tool and generate valid arguments reliably.

Two tools are defined below:

1. **Calculator** — evaluates basic arithmetic expressions.
2. **Weather lookup** — a stub that returns fixed demo data for a small set of cities instead of calling a real weather API.

The weather stub is intentional because the task asks for a simple tool implementation that demonstrates the tool-calling mechanism without requiring an external weather service.


In [19]:
CALCULATOR_TOOL = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": (
            "Evaluates a basic arithmetic expression and returns the numeric result. "
            "Use this whenever the user asks for an exact arithmetic calculation "
            "that should not be estimated or guessed. "
            "Supports numbers, +, -, *, /, **, unary +/-, and parentheses. "
            "Example input: '(4 + 5) * 2'."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": (
                        "A valid arithmetic expression using numbers, "
                        "+, -, *, /, **, and parentheses. "
                        "Example: '12 * (3 + 1)'"
                    ),
                }
            },
            "required": ["expression"],
            "additionalProperties": False,
        },
    },
}


WEATHER_TOOL = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": (
            "Looks up the weather for a given city and returns the temperature "
            "in Celsius and the weather condition. Use this whenever the user "
            "asks about weather, temperature, or wants to compare weather "
            "between cities. This is a stub for a real weather API: it returns "
            "fixed demo data for a small set of known cities and an error "
            "for unknown cities."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "City name, e.g. 'Lahore' or 'Karachi'",
                }
            },
            "required": ["city"],
            "additionalProperties": False,
        },
    },
}


TOOLS = [CALCULATOR_TOOL, WEATHER_TOOL]

print(json.dumps(TOOLS, indent=2))

[
  {
    "type": "function",
    "function": {
      "name": "calculator",
      "description": "Evaluates a basic arithmetic expression and returns the numeric result. Use this whenever the user asks for an exact arithmetic calculation that should not be estimated or guessed. Supports numbers, +, -, *, /, **, unary +/-, and parentheses. Example input: '(4 + 5) * 2'.",
      "parameters": {
        "type": "object",
        "properties": {
          "expression": {
            "type": "string",
            "description": "A valid arithmetic expression using numbers, +, -, *, /, **, and parentheses. Example: '12 * (3 + 1)'"
          }
        },
        "required": [
          "expression"
        ],
        "additionalProperties": false
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Looks up the weather for a given city and returns the temperature in Celsius and the weather condition. Use this whenever the user asks

**Why tool descriptions matter**

The model does not see your Python implementation — it sees the tool's `name`, `description`, and `parameters` provided in the tool schema. The description is an important part of the tool's interface contract: it tells the model **when** to use the tool, **what** the tool does, **what** its arguments mean, and what formats or units are expected.

A vague description such as `"does math"` gives the model little guidance and can lead to incorrect or missed tool calls. A specific description such as `"evaluates a basic arithmetic expression and supports +, -, *, /, **, and parentheses"` gives the model clearer instructions and helps reduce incorrect tool selection and malformed arguments.


In [20]:
import ast
import operator as op
import json


_FAKE_WEATHER_DB = {
    "lahore": {"temp_c": 41, "condition": "sunny"},
    "karachi": {"temp_c": 34, "condition": "humid"},
    "islamabad": {"temp_c": 33, "condition": "cloudy"},
    "sargodha": {"temp_c": 40, "condition": "sunny"},
}


# Operators allowed by the calculator
_ALLOWED_OPERATORS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.UAdd: op.pos,
    ast.USub: op.neg,
}


def run_calculator(expression: str) -> str:
    """Safely evaluate a simple arithmetic expression."""

    try:
        tree = ast.parse(expression, mode="eval")

        def evaluate(node):
            if isinstance(node, ast.Expression):
                return evaluate(node.body)

            # Numbers
            if isinstance(node, ast.Constant):
                if isinstance(node.value, (int, float)):
                    return node.value
                raise ValueError("Only numbers are allowed.")

            # Unary operators: +5, -5
            if isinstance(node, ast.UnaryOp):
                operator = _ALLOWED_OPERATORS.get(type(node.op))

                if operator is None:
                    raise ValueError("Unsupported unary operator.")

                return operator(evaluate(node.operand))

            # Binary operators: +, -, *, /, **
            if isinstance(node, ast.BinOp):
                operator = _ALLOWED_OPERATORS.get(type(node.op))

                if operator is None:
                    raise ValueError("Unsupported operator.")

                left = evaluate(node.left)
                right = evaluate(node.right)

                # Prevent excessively large exponentiation
                if isinstance(node.op, ast.Pow) and abs(right) > 100:
                    raise ValueError("Exponent is too large.")

                return operator(left, right)

            raise ValueError(
                "Only numbers, +, -, *, /, **, unary +/-, "
                "and parentheses are supported."
            )

        result = evaluate(tree)
        return str(result)

    except ZeroDivisionError:
        return "ERROR: division by zero."

    except (SyntaxError, ValueError, TypeError, OverflowError) as e:
        return f"ERROR: could not evaluate expression '{expression}': {e}"

    except Exception as e:
        return f"ERROR: calculator failed: {e}"


def run_get_weather(city: str) -> str:
    """Stub weather lookup -- deliberately fails for unknown cities."""

    data = _FAKE_WEATHER_DB.get(city.strip().lower())

    if data is None:
        return f"ERROR: no weather data available for city '{city}'"

    return json.dumps(data)


def execute_tool(name: str, tool_input: dict) -> str:
    """Dispatch a tool name to its implementation."""

    if name == "calculator":
        return run_calculator(
            tool_input.get("expression", "")
        )

    elif name == "get_weather":
        return run_get_weather(
            tool_input.get("city", "")
        )

    else:
        return f"ERROR: unknown tool '{name}'"

In [21]:
print("=== Calculator Tests ===")

test_expressions = [
    "234 * 18",
    "(50 + 10) / 2",
    "2 ** 8",
    "-5 + 12",
    "10 / 0",
    "2 + abc",
]

for expression in test_expressions:
    print(f"{expression} -> {run_calculator(expression)}")


print("\n=== Weather Tests ===")

print("Lahore:", run_get_weather("Lahore"))
print("Karachi:", run_get_weather("Karachi"))
print("Atlantis:", run_get_weather("Atlantis"))


print("\n=== Unknown Tool Test ===")

print(
    execute_tool(
        "send_email",
        {"recipient": "manager@example.com"}
    )
)

=== Calculator Tests ===
234 * 18 -> 4212
(50 + 10) / 2 -> 30.0
2 ** 8 -> 256
-5 + 12 -> 7
10 / 0 -> ERROR: division by zero.
2 + abc -> ERROR: could not evaluate expression '2 + abc': Only numbers, +, -, *, /, **, unary +/-, and parentheses are supported.

=== Weather Tests ===
Lahore: {"temp_c": 41, "condition": "sunny"}
Karachi: {"temp_c": 34, "condition": "humid"}
Atlantis: ERROR: no weather data available for city 'Atlantis'

=== Unknown Tool Test ===
ERROR: unknown tool 'send_email'


### Single tool-call round trip

In [23]:
def task2_single_tool_call_demo():
    print("=== TASK 2: single tool-call round trip ===\n")
    messages = [{"role": "user", "content": "What is 234 * 18?"}]

    response = client.chat.completions.create(
        model=MODEL,
        tools=TOOLS,
        messages=messages,
    )
    message = response.choices[0].message

    if not message.tool_calls:
        print("[REASON] Model has final answer.")
        print(f"[REASON] Model answered directly: {message.content}")
        return

    tool_call = message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    print(f"[REASON] Model chose tool: {tool_call.function.name} with input {args}")

    print(f"[ACT] Executing {tool_call.function.name}...")
    result = execute_tool(tool_call.function.name, args)
    print(f"[OBSERVE] Result: {result}")

    # Manually continue the conversation with the tool result
    messages.append(message.model_dump(exclude_unset=True))
    messages.append(
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result,
        }
    )

    final = client.chat.completions.create(model=MODEL, tools=TOOLS, messages=messages)
    final_text = final.choices[0].message.content
    print(f"[REASON] Model has final answer.")
    print(f"Final answer: {final_text}")


task2_single_tool_call_demo()


=== TASK 2: single tool-call round trip ===

[REASON] Model chose tool: calculator with input {'expression': '234 * 18'}
[ACT] Executing calculator...
[OBSERVE] Result: 4212
[REASON] Model has final answer.
Final answer: 234 × 18 = **4,212**


## Task 3: Build a Minimal Agent Loop

The agent uses an explicit `while` loop to implement the ReAct-style cycle:

**model decision → tool execution → observation → repeat**

For each iteration, the agent sends the current `messages` history to the model and checks whether the response contains a tool call. If a tool call is requested, Python parses the arguments, executes the corresponding tool, appends the tool result to the conversation history, and sends the updated history back to the model. The loop continues until the model returns a final text response or the `max_iterations` safeguard is reached.

Each iteration produces concise logs using `[MODEL]`, `[ACT]`, and `[OBSERVE]` labels. These logs show the model's observable decision/output, the tool execution, and the resulting observation without dumping the complete raw API response. This provides a simple debugging view of the agent's execution loop.


In [24]:
def run_agent(
    user_task: str,
    max_iterations: int = 6,
    verbose: bool = True
) -> str:
    """
    Minimal ReAct-style agent loop.

    Reason (model call) -> Act (tool execution)
    -> Observe (tool result) -> repeat.

    Stops when the model returns a final text answer
    or when max_iterations is reached.
    """

    messages = [{"role": "user", "content": user_task}]

    # Task 4: working memory / agent state
    working_memory = {
        "iterations": 0,
        "tool_calls": [],
        "observations": [],
        "errors": [],
    }

    # Explicit while-loop as required by the assignment
    iteration = 0

    while iteration < max_iterations:

        # Increment iteration at the beginning of each loop
        iteration += 1
        working_memory["iterations"] = iteration

        if verbose:
            print(
                f"\n--- Iteration {iteration}/{max_iterations} ---"
            )

        # --------------------------------------------------
        # 1. REASON: Ask the model what to do next
        # --------------------------------------------------
        try:
            response = client.chat.completions.create(
                model=MODEL,
                tools=TOOLS,
                messages=messages,
            )

        except Exception as e:
            error_message = f"ERROR: LLM request failed: {e}"
            working_memory["errors"].append(error_message)

            if verbose:
                print(f"[ERROR] {error_message}")

            return error_message

        message = response.choices[0].message

        if message.content and verbose:
            print(f"[MODEL] {message.content}")

        # --------------------------------------------------
        # 2. CHECK: Does the model have a final answer?
        # --------------------------------------------------
        if not message.tool_calls:

            if verbose:
                print(
                    f"[DONE] Final answer after "
                    f"{iteration} iteration(s)."
                )

            return message.content or ""

        # --------------------------------------------------
        # 3. Add assistant's tool-call message to history
        # --------------------------------------------------
        messages.append(
            message.model_dump(exclude_unset=True)
        )

        # --------------------------------------------------
        # 4. ACT: Execute each requested tool
        # --------------------------------------------------
        for tc in message.tool_calls:

            tool_name = tc.function.name

            # Parse tool arguments safely
            try:
                args = json.loads(tc.function.arguments)

            except json.JSONDecodeError as e:

                result = (
                    f"ERROR: invalid JSON tool arguments "
                    f"for '{tool_name}': {e}"
                )

                working_memory["errors"].append(result)

                if verbose:
                    print(f"[ERROR] {result}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                })

                continue

            # Log tool decision
            if verbose:
                print(
                    f"[ACT] Calling tool '{tool_name}' "
                    f"with input {args}"
                )

            working_memory["tool_calls"].append({
                "iteration": iteration,
                "tool": tool_name,
                "input": args,
            })

            # Execute tool safely
            try:
                result = execute_tool(
                    tool_name,
                    args
                )

            except Exception as e:

                result = (
                    f"ERROR: tool '{tool_name}' "
                    f"failed: {e}"
                )

                working_memory["errors"].append(result)

            # --------------------------------------------------
            # 5. OBSERVE: Record the tool result
            # --------------------------------------------------
            if verbose:
                print(f"[OBSERVE] Result: {result}")

            working_memory["observations"].append({
                "iteration": iteration,
                "tool": tool_name,
                "input": args,
                "result": result,
            })

            # --------------------------------------------------
            # 6. Return observation to the model
            # --------------------------------------------------
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

    # ------------------------------------------------------
    # 7. Safeguard: maximum iterations reached
    # ------------------------------------------------------
    if verbose:
        print(
            f"\n[STOPPED] Hit max_iterations="
            f"{max_iterations} safeguard without "
            f"a final answer."
        )

    return (
        "AGENT STOPPED: max_iterations reached "
        "without a final answer."
    )

### Test on a multi-step task (requires 2+ tool calls)

In [25]:
print("=== TASK 3: multi-step agent loop (2+ tool calls) ===")
answer = run_agent(
    "Look up the weather in Lahore and Karachi, then tell me which city is warmer "
    "and by how many degrees."
)
print(f"\nAgent's final answer: {answer}")


=== TASK 3: multi-step agent loop (2+ tool calls) ===

--- Iteration 1/6 ---
[ACT] Calling tool 'get_weather' with input {'city': 'Lahore'}
[OBSERVE] Result: {"temp_c": 41, "condition": "sunny"}

--- Iteration 2/6 ---
[ACT] Calling tool 'get_weather' with input {'city': 'Karachi'}
[OBSERVE] Result: {"temp_c": 34, "condition": "humid"}

--- Iteration 3/6 ---
[MODEL] **Weather comparison**

| City | Temperature (°C) | Condition |
|------|------------------|-----------|
| Lahore | 41 °C | Sunny |
| Karachi | 34 °C | Humid |

**Result:** Lahore is warmer than Karachi by **7 °C**.
[DONE] Final answer after 3 iteration(s).

Agent's final answer: **Weather comparison**

| City | Temperature (°C) | Condition |
|------|------------------|-----------|
| Lahore | 41 °C | Sunny |
| Karachi | 34 °C | Humid |

**Result:** Lahore is warmer than Karachi by **7 °C**.


## Task 4: Memory & State Handling

### Conversation Memory vs Working Memory

* **Conversation memory** = the `messages` list maintained by the Python agent. It contains the user message, assistant responses, tool-call messages, and tool-result messages. This conversation history is sent back to the model on each API call, allowing the model to use information from earlier steps. The API itself does not automatically maintain conversation state between separate requests; the agent must provide the relevant history.

* **Working memory** = state maintained by the Python agent outside the conversation history. It can track information about the agent's execution, such as the current iteration, tools that have been called, observations returned by tools, and errors encountered during execution. In `run_agent`, the `working_memory` dictionary serves as this execution-state scratchpad. In this implementation, it is primarily used for tracking and debugging and is not automatically sent to the model.

The distinction matters because conversation history can grow as more messages and tool results are added, increasing token usage and eventually approaching context limits. Working memory can instead be structured or pruned according to the task—for example, retaining only the most important tool results or execution state.

The logging in `run_agent` provides a simple debugging view of the agent loop using `[MODEL]`, `[ACT]`, `[OBSERVE]`, `[ERROR]`, and `[DONE]` labels. When an agent behaves unexpectedly, these logs help identify what the model requested, which tool was executed, and what result was returned. Frameworks provide higher-level abstractions around this same underlying state-and-tool loop.


## Task 5: Failure Modes & Guardrails

Deliberately testing the agent under failure and uncertainty conditions:

1. **Unknown city / tool error** — request weather for `Atlantis`, causing the weather tool to return an error that the agent must handle.
2. **Unsupported capability** — request an operation for which no corresponding tool is defined. The model should recognize that the required capability is unavailable rather than inventing a tool or result.
3. **Ambiguous request** — provide an underspecified request such as `"Is it nice outside?"` and observe whether the agent asks for the missing location instead of guessing.

These tests demonstrate how the agent handles tool errors, unavailable capabilities, and ambiguous user requests.


In [27]:
def task5_break_it_deliberately():
    print("=== TASK 5: deliberately breaking the agent ===")

    # --------------------------------------------------
    # Break 1: Tool returns an error
    # --------------------------------------------------
    print("\n-- Break 1: tool returns an error (unknown city) --")

    run_agent(
        "Use the get_weather tool to look up the weather "
        "for Atlantis. Do not answer from your own knowledge.",
        max_iterations=3
    )

    # --------------------------------------------------
    # Break 2: Undefined capability/tool
    # --------------------------------------------------
    print("\n-- Break 2: task requiring an undefined tool --")

    run_agent(
        "I need you to convert 100 USD to PKR. "
        "You must use a currency_conversion tool to perform "
        "the conversion. Do not calculate it yourself.",
        max_iterations=3
    )

    # --------------------------------------------------
    # Break 3: Ambiguous request
    # --------------------------------------------------
    print("\n-- Break 3: ambiguous request --")

    run_agent(
        "Is it nice outside?",
        max_iterations=3
    )


task5_break_it_deliberately()

=== TASK 5: deliberately breaking the agent ===

-- Break 1: tool returns an error (unknown city) --

--- Iteration 1/3 ---
[ACT] Calling tool 'get_weather' with input {'city': 'Atlantis'}
[OBSERVE] Result: ERROR: no weather data available for city 'Atlantis'

--- Iteration 2/3 ---
[MODEL] I’m sorry, but I couldn’t find any weather data for **Atlantis**. The weather service only contains information for a limited set of real-world cities. If you have another location in mind, let me know and I’ll look it up!
[DONE] Final answer after 2 iteration(s).

-- Break 2: task requiring an undefined tool --

--- Iteration 1/3 ---
[MODEL] I don’t have a dedicated currency‑conversion tool available in this environment, so I can’t perform the conversion with a verified API. If you have a reliable exchange‑rate source (e.g., a banking API, a finance website, or a spreadsheet), you can use it to get the current USD→PKR rate. Once you have that rate, just multiply it by 100 to get the amount in PKR.
[

## Observed Failure Modes & Mitigations

1. **Infinite / near-infinite loop** — the model could continue requesting tools without converging on a final answer.

   **Mitigation:** The agent uses a `max_iterations` safeguard. If the loop reaches the maximum number of iterations without producing a final answer, it stops and returns an explicit `"AGENT STOPPED"` message. A clear instruction to provide a final answer once enough information has been gathered can also reduce unnecessary iterations.

2. **Unsupported capability / unavailable tool** — the user may request an action for which no corresponding tool has been defined. In the experiment, the currency-conversion request did not produce a nonexistent tool call; instead, the model recognized that the required capability was unavailable.

   **Mitigation:** Only tools explicitly supplied in the `TOOLS` list can be executed. The `execute_tool()` dispatch function also contains an unknown-tool branch that returns an explicit error instead of crashing if an unexpected tool name reaches the executor.

3. **Wrong or malformed tool arguments** — a model could provide incorrectly formatted JSON, an inappropriate value such as an invalid city name, or an invalid arithmetic expression.

   **Mitigation:** Tool schemas define the expected argument structure using JSON Schema. The agent additionally parses tool arguments with `json.loads()` and catches `JSONDecodeError`. The calculator performs its own input validation and returns a clear error instead of silently producing an incorrect result.

4. **Tool errors / unhandled exceptions** — a tool may fail because of invalid input, unavailable data, or an unexpected runtime error.

   **Mitigation:** Tool execution is wrapped in `try/except`. Errors are converted into readable strings and returned to the model as `tool` messages, allowing the model to observe the failure and decide what to do next. The agent also records errors in `working_memory`.

5. **Ambiguous request → incorrect assumption** — a request such as `"Is it nice outside?"` does not specify a city or location. The model could guess, answer generically, or ask for clarification. In the experiment, the model correctly asked which city or location the user meant.

   **Mitigation:** Tool descriptions should clearly specify required inputs. When essential information is missing, the agent should ask a clarifying question rather than inventing a value.

6. **Context / token growth during long loops** — every assistant tool call and tool result is appended to `messages`. A long-running agent can therefore accumulate a large conversation history and eventually approach the model's context limit.

   **Mitigation:** For longer-running agents, old tool results can be summarized or pruned while retaining the most important state in a compact working-memory structure.

### Why do frameworks like LangChain, LangGraph, and CrewAI exist?

The raw implementation demonstrates that an agent is fundamentally a loop involving model decisions, tool execution, observations, state management, error handling, and stopping conditions. However, writing and maintaining all of these components manually becomes increasingly complex as agents grow.

Frameworks such as LangChain, LangGraph, and CrewAI provide higher-level abstractions for common requirements such as tool management, structured state, retries, workflow/graph execution, multi-agent coordination, memory, and observability. They can also reduce the amount of provider-specific code required when working with different LLM APIs.

Building the agent from scratch first is useful because it makes these abstractions easier to understand: concepts such as nodes, edges, executors, tools, state, and agent loops are abstractions over mechanisms that have already been implemented manually in this exercise.
